Supervised Fine Tuning on Open Source LLM - Qwen 0.5B

In [ ]:
# Environment Setup

import os
import sys
import subprocess

IS_COLAB = "google.colab" in sys.modules

REPO_NAME = "RLHF_Implementation"
REPO_URL = "https://github.com/Samarth737/RLHF_Implementation.git"
REPO_PATH = f"/content/{REPO_NAME}"

def run(cmd, allow_fail=False):
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.returncode != 0 and not allow_fail:
        print(result.stderr)
        result.check_returncode()
    return result

if IS_COLAB:
    print("Running in Google Colab")

    if not os.path.exists(REPO_PATH):
        print("Cloning repository...")
        run(["git", "clone", REPO_URL, REPO_PATH])

    os.chdir(REPO_PATH)
    print("Working directory:", os.getcwd())

    if not os.path.exists("/content/.deps_installed"):
        print("Installing dependencies...")

        run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

        install_ok = False
        if os.path.exists("requirements.txt"):
            result = run(
                [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
                allow_fail=True
            )
            install_ok = (result.returncode == 0)

            if not install_ok:
                print("requirements.txt failed in Colab, installing Colab-safe packages instead...")

        if not install_ok:
            run([
                sys.executable, "-m", "pip", "install",
                "torch", "torchvision", "torchaudio",
                "transformers", "accelerate", "datasets", "trl", "peft",
                "sentencepiece", "scipy", "matplotlib", "pandas", "numpy"
            ])

            run([
                sys.executable, "-m", "pip", "install", "bitsandbytes"
            ], allow_fail=True)

        open("/content/.deps_installed", "w").close()
        print("Dependency setup complete.")

else:
    print("Running locally")
    print("Working directory:", os.getcwd())

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
import sys
import torch

print("Python:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Python: c:\Users\samar\RLHF_Impl\venv\Scripts\python.exe
Torch version: 2.10.0+cu126
CUDA available: True
Torch CUDA version: 12.6
Device count: 1


In [3]:
import sys
print(sys.executable)

c:\Users\samar\RLHF_Impl\venv\Scripts\python.exe


In [4]:
import os
import math
import random
from dataclasses import dataclass
from typing import Dict, List

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
DATASET_NAME = "tatsu-lab/alpaca"
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 1
NUM_EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
TRAIN_VAL_SPLIT = 0.1
SEED = 42
LOG_EVERY = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = "./sft/checkpoints/qwen"

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16


Prompt formatting

In [5]:
def format_prompt(example: Dict[str, str]) -> str:
    instruction = (example.get("instruction") or "").strip()
    input_text = (example.get("input") or "").strip()

    if input_text:
        return (
            "Below is an instruction that describes a task, paired with an input that provides further context.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            "### Response:\n"
        )
    else:
        return (
            "Below is an instruction that describes a task.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            "### Response:\n"
        )
    
def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

Dataset:

    This class creates:
      input_ids :- prompt + response + eos
      attention_mask :- standard mask
      labels :- -100 for prompt tokens, actual ids for response tokens

In [6]:
class AlpacaSFTDataset(Dataset):

    def __init__(self, hf_dataset, tokenizer, max_length: int):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        ex = self.dataset[idx]

        prompt = format_prompt(ex)
        response = (ex.get("output") or "").strip()

        # Add EOS to the target completion
        full_text = prompt + response + self.tokenizer.eos_token

        # Tokenizing full sequence
        full_enc = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
            return_attention_mask=True,
        )

        # Tokenize prompt alone to find label masking boundary
        prompt_enc = self.tokenizer(
            prompt,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
            return_attention_mask=False,
        )

        input_ids = full_enc["input_ids"]
        attention_mask = full_enc["attention_mask"]

        prompt_len = len(prompt_enc["input_ids"])
        # Mask prompt tokens and only learn on response tokens
        labels = [-100] * len(input_ids)
        for i in range(prompt_len, len(input_ids)):
            labels[i] = input_ids[i]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

Collator

In [7]:

@dataclass
class SFTCollator:
    tokenizer: AutoTokenizer

    def __call__(self, batch: List[Dict[str, List[int]]]) -> Dict[str, torch.Tensor]:
        pad_id = self.tokenizer.pad_token_id
        if pad_id is None:
            raise ValueError("Tokenizer pad_token_id is None. Set tokenizer.pad_token first.")

        max_len = max(len(x["input_ids"]) for x in batch)

        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []

        for x in batch:
            seq_len = len(x["input_ids"])
            pad_len = max_len - seq_len

            batch_input_ids.append(x["input_ids"] + [pad_id] * pad_len)
            batch_attention_mask.append(x["attention_mask"] + [0] * pad_len)
            batch_labels.append(x["labels"] + [-100] * pad_len)

        return {
            "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
        }



Evaluation and Metrics


In [8]:

#Causal LM predicts token t using position t-1 logits.
#So compare logits[:, :-1, :] against labels[:, 1:].
#Ignore labels == -100.
@torch.no_grad()
def compute_token_accuracy(logits: torch.Tensor, labels: torch.Tensor) -> float:
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    preds = shift_logits.argmax(dim=-1)
    mask = shift_labels != -100

    if mask.sum().item() == 0:
        return 0.0

    correct = (preds[mask] == shift_labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total


@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_batches = 0

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )

        #print(f"Sample logits:{outputs.logits[0,0,0]}")

        loss = outputs.loss
        acc = compute_token_accuracy(outputs.logits, batch["labels"])

        total_loss += loss.item()
        total_acc += acc
        total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    avg_acc = total_acc / max(total_batches, 1)
    perplexity = math.exp(min(avg_loss, 20))  # avoid overflow

    return {
        "loss": avg_loss,
        "perplexity": perplexity,
        "token_accuracy": avg_acc,
    }


In [9]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def plot_sft_metrics(
    output_dir,
    train_steps,
    train_losses,
    val_epochs,
    val_losses,
    val_ppls,
    val_accs
):

    plot_dir = os.path.join(output_dir, "plots")
    os.makedirs(plot_dir, exist_ok=True)

    # Save CSV files
    pd.DataFrame({
        "train_step": train_steps,
        "train_loss": train_losses
    }).to_csv(os.path.join(plot_dir, "sft_train_metrics.csv"), index=False)

    pd.DataFrame({
        "epoch": val_epochs,
        "val_loss": val_losses,
        "val_perplexity": val_ppls,
        "val_token_accuracy": val_accs
    }).to_csv(os.path.join(plot_dir, "sft_val_metrics.csv"), index=False)

    # Training loss plot
    plt.figure(figsize=(8,5))
    plt.plot(train_steps, train_losses, marker="o")
    plt.xlabel("Training Step")
    plt.ylabel("Train Loss")
    plt.title("SFT Training Loss")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "sft_train_loss.png"), dpi=300)
    plt.close()

    # Validation loss plot
    plt.figure(figsize=(8,5))
    plt.plot(val_epochs, val_losses, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Validation Loss")
    plt.title("SFT Validation Loss")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "sft_val_loss.png"), dpi=300)
    plt.close()

    # Perplexity plot
    plt.figure(figsize=(8,5))
    plt.plot(val_epochs, val_ppls, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Validation Perplexity")
    plt.title("SFT Validation Perplexity")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "sft_val_perplexity.png"), dpi=300)
    plt.close()

    # Accuracy plot
    plt.figure(figsize=(8,5))
    plt.plot(val_epochs, val_accs, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Token Accuracy")
    plt.title("SFT Validation Token Accuracy")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "sft_val_token_accuracy.png"), dpi=300)
    plt.close()

    print("Saved SFT plots to:", plot_dir)

Training:

In [10]:

def main():
    set_seed(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"Using device: {DEVICE}")
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)

    # Many causal LMs don't define a pad token. For training, using EOS as PAD is practical.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.to(DEVICE)

    print("Loading dataset...")
    raw_ds = load_dataset(DATASET_NAME, split="train[:1000]")

    split_ds = raw_ds.train_test_split(test_size=TRAIN_VAL_SPLIT, seed=SEED)
    train_ds_raw = split_ds["train"]
    val_ds_raw = split_ds["test"]

    print(f"Train size: {len(train_ds_raw)}")
    print(f"Val size:   {len(val_ds_raw)}")

    train_ds = AlpacaSFTDataset(train_ds_raw, tokenizer, MAX_LENGTH)
    val_ds = AlpacaSFTDataset(val_ds_raw, tokenizer, MAX_LENGTH)

    print(f"Sample dataset: {raw_ds[0]}")

    collator = SFTCollator(tokenizer)

    train_loader = DataLoader(
        train_ds,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    total_update_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * NUM_EPOCHS
    warmup_steps = int(total_update_steps * WARMUP_RATIO)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_update_steps,
    )

    scaler = torch.amp.GradScaler(enabled=USE_FP16)

    best_val_loss = float("inf")
    global_step = 0

    train_loss_history = []
    train_step_history = []

    val_loss_history = []
    val_ppl_history = []
    val_acc_history = []
    val_epoch_history = []

    print("Starting training...")
    for epoch in range(NUM_EPOCHS):
        model.train()
        optimizer.zero_grad()

        running_loss = 0.0

        for step, batch in enumerate(train_loader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16 if USE_BF16 else torch.float16,
                enabled=(USE_BF16 or USE_FP16),
            ):
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["labels"],
                )
                loss = outputs.loss / GRAD_ACCUM_STEPS

            if USE_FP16:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            running_loss += loss.item() * GRAD_ACCUM_STEPS

            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                if USE_FP16:
                    scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                if USE_FP16:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()

                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % LOG_EVERY == 0:
                    avg_train_loss = running_loss / LOG_EVERY
                    current_lr = scheduler.get_last_lr()[0]

                    train_step_history.append(global_step)
                    train_loss_history.append(avg_train_loss)

                    print(
                        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
                        f"Step {global_step} | "
                        f"Train Loss: {avg_train_loss:.4f} | "
                        f"LR: {current_lr:.6e}"
                    )
                    running_loss = 0.0

        # End-of-epoch evaluation
        val_metrics = evaluate(model, val_loader, DEVICE)
        val_epoch_history.append(epoch + 1)
        val_loss_history.append(val_metrics["loss"])
        val_ppl_history.append(val_metrics["perplexity"])
        val_acc_history.append(val_metrics["token_accuracy"])
        print(
            f"\n[Epoch {epoch + 1}] "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val PPL: {val_metrics['perplexity']:.4f} | "
            f"Val Token Acc: {val_metrics['token_accuracy']:.4f}\n"
        )

        # Save best checkpoint
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_dir = os.path.join(OUTPUT_DIR, "best")
            os.makedirs(best_dir, exist_ok=True)

            model.save_pretrained(best_dir)
            tokenizer.save_pretrained(best_dir)

            torch.save(
                {
                    "epoch": epoch + 1,
                    "best_val_loss": best_val_loss,
                    "model_name": MODEL_NAME,
                    "max_length": MAX_LENGTH,
                },
                os.path.join(best_dir, "training_meta.pt"),
            )
            print(f"Saved best model to: {best_dir}")

    # Save final checkpoint
    final_dir = os.path.join(OUTPUT_DIR, "final")
    os.makedirs(final_dir, exist_ok=True)
    model.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)

    torch.save(
        {
            "epochs": NUM_EPOCHS,
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
        },
        os.path.join(final_dir, "training_meta.pt"),
    )

    plot_sft_metrics(
    OUTPUT_DIR,
    train_step_history,
    train_loss_history,
    val_epoch_history,
    val_loss_history,
    val_ppl_history,
    val_acc_history
    )

    print(f"Training complete. Final model saved to: {final_dir}")

if __name__ == "__main__":
    main()


Using device: cuda
Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading dataset...
Train size: 900
Val size:   100
Sample dataset: {'instruction': 'Give three tips for staying healthy.', 'input': '', 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}
Starting training...
Epoch 1/2 | Step 10 | Train Loss: 1.6172 | LR: 1.538462e-05
Epoch 1/2 | Step 20 | Train Loss: 1.3485 | LR: 1.967963e-05
Epoch 1/2 | Step 30 | Train Loss: 1.6202 | LR: 1.922197e-05
Epoch 1/2 | Step 40 | Train Loss: 1.6002 | LR: 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model to: ./sft/checkpoints/qwen\best
Epoch 2/2 | Step 230 | Train Loss: 0.6081 | LR: 1.006865e-05
Epoch 2/2 | Step 240 | Train Loss: 1.1657 | LR: 9.610984e-06
Epoch 2/2 | Step 250 | Train Loss: 1.1394 | LR: 9.153318e-06
Epoch 2/2 | Step 260 | Train Loss: 0.9893 | LR: 8.695652e-06
Epoch 2/2 | Step 270 | Train Loss: 1.1729 | LR: 8.237986e-06
Epoch 2/2 | Step 280 | Train Loss: 1.0755 | LR: 7.780320e-06
Epoch 2/2 | Step 290 | Train Loss: 1.1239 | LR: 7.322654e-06
Epoch 2/2 | Step 300 | Train Loss: 1.1521 | LR: 6.864989e-06
Epoch 2/2 | Step 310 | Train Loss: 1.1091 | LR: 6.407323e-06
Epoch 2/2 | Step 320 | Train Loss: 1.0710 | LR: 5.949657e-06
Epoch 2/2 | Step 330 | Train Loss: 1.1896 | LR: 5.491991e-06
Epoch 2/2 | Step 340 | Train Loss: 1.4423 | LR: 5.034325e-06
Epoch 2/2 | Step 350 | Train Loss: 1.2784 | LR: 4.576659e-06
Epoch 2/2 | Step 360 | Train Loss: 1.0068 | LR: 4.118993e-06
Epoch 2/2 | Step 370 | Train Loss: 1.2219 | LR: 3.661327e-06
Epoch 2/2 | Step 380 | Train Loss: 1

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved SFT plots to: ./sft/checkpoints/qwen\plots
Training complete. Final model saved to: ./sft/checkpoints/qwen\final


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_DIR = "./sft/checkpoints/qwen/best"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

prompt = (
    "Below is an instruction that describes a task.\n\n"
    "### Instruction:\n"
    "Explain what RLHF is in simple terms.\n\n"
    "### Response:\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Below is an instruction that describes a task.

### Instruction:
Explain what RLHF is in simple terms.

### Response:
RLHF stands for Receptive Learning in High Dimensions. It is a form of reinforcement learning where the agent is provided with a set of instructions that are either positive or negative. It is then trained on the data that it receives to learn how to make decisions in high dimensions.


Comparison of responses before and after Supervised Fine Tuning

In [12]:
BASE_MODEL = MODEL_NAME
SFT_MODEL = "./sft/checkpoints/qwen/best"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading BASE model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
).to(device)

print("Loading SFT model...")
sft_model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL,
    trust_remote_code=True
).to(device)

base_model.eval()
sft_model.eval()

Loading tokenizer...
Loading BASE model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading SFT model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896, padding_idx=151643)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
   

In [13]:
prompt = """
Below is an instruction that describes a task.

### Instruction:
How to be efficient?

### Response:
"""
inputs = tokenizer(prompt, return_tensors="pt").to(device)
def generate(model):
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


print("\n===== BASE MODEL =====\n")
print(generate(base_model))

print("\n===== SFT MODEL =====\n")
print(generate(sft_model))


===== BASE MODEL =====


Below is an instruction that describes a task.

### Instruction:
How to be efficient?

### Response:
To be efficient in the context of this instruction, you should focus on using the most effective and practical methods to achieve your goals. This may involve breaking down complex tasks into smaller, more manageable steps, utilizing efficient algorithms or tools, and making use of techniques that can speed up or simplify calculations. Additionally, being mindful of your time and energy consumption, and finding ways to optimize your workflow and minimize distractions, can also contribute to being efficient. Remember that efficiency is not just about speed, but also about accuracy, efficiency, and effectiveness in achieving your objectives.

===== SFT MODEL =====


Below is an instruction that describes a task.

### Instruction:
How to be efficient?

### Response:
1. Know your goals and objectives.
2. Set a schedule for yourself.
3. Create a to-do list and prior